# Summer Storm Langlois HI

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os 
from scipy.optimize import curve_fit
from scipy.integrate import quad

storm_directory = '../../data/constituents'
storms = {}
for filename in os.listdir(storm_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(storm_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        storms[key] = df                                    # store the data frame in the dictionary

shear_stress = pd.read_csv('../../data/shear_stress/average_total_shear_stress_corrected.csv', parse_dates=['datetime'], index_col='datetime')
sonde_downstream = pd.read_csv('../../data/sonde_data/sonde_down_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')
sonde_upstream = pd.read_csv('../../data/sonde_data/sonde_up_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')

Process and Merge Data

In [3]:
# average rows with duplicate timestamps on sonde data (second-resolution data)
sonde_downstream = sonde_downstream.groupby(level=0).mean(numeric_only=True)
sonde_upstream = sonde_upstream.groupby(level=0).mean(numeric_only=True)

# re-sample shear stress and sonde data to 1 min intervals
shear_stress = shear_stress.resample('1min').interpolate()
sonde_downstream_resampled = sonde_downstream.resample('1min').interpolate()
sonde_upstream_resampled = sonde_upstream.resample('1min').interpolate()

In [4]:
# join shear stress data and the matching sonde data with the storm data
merged_storms = {}

for storm_name, storm_df in storms.items():
    if "down" in storm_name.lower():
        sonde_df = sonde_downstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    elif "up" in storm_name.lower():
        sonde_df = sonde_upstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    else:
        continue

    merged_storm_df = storm_df.join(shear_stress, how="left")
    merged_storm_df = merged_storm_df.join(sonde_df, how="left")
    merged_storms[storm_name] = merged_storm_df

# also join the sonde data with the shear stress data for the entire record 
sonde_downstream = sonde_downstream.join(shear_stress['shear_stress'], how="left")
sonde_upstream = sonde_upstream.join(shear_stress['shear_stress'], how="left")

In [8]:
merged_storms['st1_down']

,SS (uL/L),SSC (mg/L),SRP (mg/L),TP (mg/L),PP (mg/L),POC (mg/L),DOC (mg/L),shear_stress,Turbidity FNU,fDOM RFU
Date_Time,,,,,,,,,,
2021-07-23 14:30:00,0.00000,4.409565,0.02625,0.03125,0.00,2.864641,1.754,68.223515,5.204000,22.8860
2021-07-23 15:50:00,339.78248,562.325800,0.06200,0.36000,1.19,51.660000,9.180,113.726185,92.530400,25.1484
2021-07-23 16:09:00,457.12000,528.034800,0.08800,0.44500,1.63,50.228000,12.630,124.882455,197.947760,25.1140
2021-07-23 16:40:00,152.79000,143.169000,0.08800,0.24800,0.77,14.502000,13.390,131.895386,78.192980,37.8446
2021-07-23 16:55:00,168.45000,82.761400,0.08300,0.21400,0.50,11.681000,13.000,129.521674,74.057744,40.2548
2021-07-23 19:19:00,43.15000,36.920000,0.05300,0.09700,0.10,6.411000,12.870,96.345185,20.570200,52.5872
2021-07-23 21:02:00,17.70000,22.011900,0.04700,0.07700,0.07,4.805000,11.550,86.958069,16.971200,54.2484


Hysteresis index calculation functions

In [9]:
## regression equations
# linear
def linear_func(Q, a, b):
    return a * Q + b
# logarithmic
def log_func(Q, a, b):
    return a * np.log(Q) + b
# exponential
def exp_func(Q, a, b):
    return a * np.exp(b * Q)

# split hydrograph into rising and falling limbs based on peak flow
def split_hydrograph(df, q_col):
    # if df empty or q_col has no valid values, return empty limbs
    if df.empty or df[q_col].dropna().empty:
        return df.copy(), df.copy()
    peak_time = df[q_col].idxmax()
    rising = df.loc[:peak_time].copy()
    falling = df.loc[peak_time:].copy()
    return rising, falling

# fit curves and calculate R²
def fit_best_curve(x, y):
    candidate_functions = {
        'linear': (linear_func, [1, 1]),
        'log': (log_func, [1, 1]),
        'exponential': (exp_func, [1, -0.01])
    }
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    mask = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x[mask])
    y = np.asarray(y[mask])

    # minimum points
    if len(x) < 2:
        return None
    best_r2 = -np.inf
    best_result = None
    # try each function and keep the one with the best r2
    for func_name, (func, p0) in candidate_functions.items():
        try:
            # avoid invalid log fits
            if func_name == 'log' and np.any(x <= 0):
                continue
            popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
            y_pred = func(x, *popt) # predicted values
            # residuals
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            # avoid divide-by-zero
            if np.isclose(ss_tot, 0):
                r2 = np.nan
            else:
                r2 = 1 - (ss_res / ss_tot)
            # keep best fit
            if np.isfinite(r2) and r2 > best_r2:
                best_r2 = r2
                best_result = {
                    'function_name': func_name,
                    'function': func,
                    'params': popt,
                    'r2': r2
                }
        except Exception:
            continue
    return best_result

# Langlois 2025 H calculation
def compute_langlois_H(event_df, tau_col, constituent_col, r2_threshold=0.50, storm_name=None):
    # data cleanup
    df = event_df[[tau_col, constituent_col]].dropna()
    if df.empty or df[tau_col].dropna().empty or df[constituent_col].dropna().empty:
        print(f"No valid {tau_col} or {constituent_col} for {storm_name or 'unknown'}; skipping")
        return None

    rising, falling = split_hydrograph(df, tau_col)
    if rising.empty or falling.empty:
        print(f"No rising or falling limb for {storm_name or 'unknown'}; skipping")
        return None

    # shear stress overlap range
    tau_min = max(rising[tau_col].min(), falling[tau_col].min())
    tau_max = min(rising[tau_col].max(), falling[tau_col].max())

    # fit rising limb
    rise_fit = fit_best_curve(rising[tau_col].values, rising[constituent_col].values)
    if rise_fit is None:
        print(f"Could not fit rising limb in " f"{storm_name or 'unknown'} for {constituent_col}")
        return None
    # fit falling limb
    fall_fit = fit_best_curve(falling[tau_col].values, falling[constituent_col].values)
    if fall_fit is None:
        print(f"Could not fit falling limb in " f"{storm_name or 'unknown'} for {constituent_col}")
        return None

    # check fit quality with r2 threshold
    if (rise_fit['r2'] < r2_threshold) or (fall_fit['r2'] < r2_threshold):
            label = storm_name if storm_name is not None else "unknown storm"
            print(f"Poor fit for rising (R²={rise_fit['r2']:.2f}) or falling (R²={fall_fit['r2']:.2f}) limb in {label} for {constituent_col}")
    # integrated areas
    rise_area, _ = quad(lambda q: rise_fit["function"](q, *rise_fit["params"]), tau_min, tau_max)
    fall_area, _ = quad(lambda q: fall_fit["function"](q, *fall_fit["params"]), tau_min, tau_max)
    # hysteresis index
    H = rise_area / fall_area

    return {
        # hysteresis
        'H': H,
        # rising limb
        'rise_r2': rise_fit['r2'],
        'rise_function': rise_fit['function'],
        'rise_params': rise_fit['params'],
        'rise_area': rise_area,
        # falling limb
        'fall_r2': fall_fit['r2'],
        'fall_function': fall_fit['function'],
        'fall_params': fall_fit['params'],
        'fall_area': fall_area,
        # overlap range
        'tau_min': tau_min,
        'tau_max': tau_max,
        # point counts
        'n_rising': len(rising),
        'n_falling': len(falling)
    }

Calculate H for all events

In [ ]:
all_results = []

for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)","Turbidity FNU"]:
        if constituent not in storm_df.columns:
            continue

        result = compute_langlois_H(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('summer_storms/langlois_hysteresis_summer.csv', index=False)

Poor fit for rising (R²=0.13) or falling (R²=0.79) limb in SP23_down_constituents for SSC (mg/L)
Poor fit for rising (R²=0.00) or falling (R²=0.50) limb in SP23_down_constituents for Turbidity FNU
Poor fit for rising (R²=0.14) or falling (R²=0.87) limb in SP23_up_constituents for SSC (mg/L)
Poor fit for rising (R²=0.01) or falling (R²=0.64) limb in SP23_up_constituents for Turbidity FNU
Poor fit for rising (R²=0.36) or falling (R²=0.85) limb in st1_down for SSC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.94) limb in st1_up for SSC (mg/L)
Poor fit for rising (R²=0.40) or falling (R²=0.99) limb in st2_down for SSC (mg/L)
Poor fit for rising (R²=0.30) or falling (R²=1.00) limb in st2_down for Turbidity FNU
Poor fit for rising (R²=0.16) or falling (R²=0.03) limb in st4_down for SSC (mg/L)
Poor fit for rising (R²=0.36) or falling (R²=0.52) limb in st4_up for SSC (mg/L)
Could not fit falling limb in st5_down for SSC (mg/L)
Could not fit falling limb in st5_down for Turbidity FNU
Po

C:\Users\nicol\AppData\Local\Temp\ipykernel_37052\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


### Plots

In [12]:
def plot_langlois_hysteresis(event_df, tau_col, constituent_col, r2_threshold=0.5, storm_name=None, 
                            out_dir='plots', save=True, show=False):

    df = event_df[[tau_col, constituent_col]].dropna()
    if df.empty:
        return None
    rising, falling = split_hydrograph(df, tau_col)
    if rising.empty or falling.empty:
        return None
    
    # reuse the same fitting logic as the H calculation
    result = compute_langlois_H(
        event_df,
        tau_col=tau_col,
        constituent_col=constituent_col,
        r2_threshold=r2_threshold,
        storm_name=storm_name,
    )
    if result is None:
        return None

    tau_min = result["tau_min"]
    tau_max = result["tau_max"]
    if not np.isfinite(tau_min) or not np.isfinite(tau_max) or tau_min >= tau_max:
        return None

    tau_fit = np.linspace(tau_min, tau_max, 200)

    rise_params = result["rise_params"]
    fall_params = result["fall_params"]
    rise_r2 = result["rise_r2"]
    fall_r2 = result["fall_r2"]
    H = result["H"]

    rise_fit = result["rise_function"](tau_fit, *result["rise_params"])
    fall_fit = result["fall_function"](tau_fit, *result["fall_params"])

    # PLOT 
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # time series
    ax = axes[0]
    ax.plot(df.index, df[tau_col], color="tab:blue", linewidth=1.5, label=tau_col)
    ax.set_ylabel(tau_col, color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.xaxis.set_major_locator(plt.MaxNLocator(8))
    ax2 = ax.twinx()
    ax2.plot(df.index, df[constituent_col], color="tab:red", linewidth=1.5, label=constituent_col)
    ax2.set_ylabel(constituent_col, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    ax.set_title("Event time series")

    # hysteresis loop
    ax = axes[1]
    ax.scatter(rising[tau_col], rising[constituent_col], label='Rising limb', color='tab:orange')
    ax.scatter(falling[tau_col], falling[constituent_col], label='Falling limb', color='tab:green')
    ax.plot(tau_fit, rise_fit, linewidth=2, label=f'Rising fit (R²={rise_r2:.2f})', color='tab:orange')
    ax.plot(tau_fit, fall_fit, linewidth=2, label=f'Falling fit (R²={fall_r2:.2f})', color='tab:green')

    ax.set_xlabel(tau_col)
    ax.set_ylabel(constituent_col)
    ax.set_title(f'H = {H:.2f}')
    ax.legend()

    # add a main title for the whole figure
    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    plt.suptitle(main_title, fontsize=15)
    plt.tight_layout()

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = f"{storm_name}_{constituent_col}_langlois.png".replace(" ", "_").replace("/", "_")
        fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    plt.close(fig)
    return result

In [13]:
all_results = []

for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)", "Turbidity FNU"]:
        if constituent not in storm_df.columns:
            continue

        plot_langlois_hysteresis(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

Poor fit for rising (R²=0.13) or falling (R²=0.79) limb in SP23_down_constituents for SSC (mg/L)


C:\Users\nicol\AppData\Local\Temp\ipykernel_37052\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Poor fit for rising (R²=0.00) or falling (R²=0.50) limb in SP23_down_constituents for Turbidity FNU
Poor fit for rising (R²=0.14) or falling (R²=0.87) limb in SP23_up_constituents for SSC (mg/L)
Poor fit for rising (R²=0.01) or falling (R²=0.64) limb in SP23_up_constituents for Turbidity FNU
Poor fit for rising (R²=0.36) or falling (R²=0.85) limb in st1_down for SSC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.94) limb in st1_up for SSC (mg/L)
Poor fit for rising (R²=0.40) or falling (R²=0.99) limb in st2_down for SSC (mg/L)
Poor fit for rising (R²=0.30) or falling (R²=1.00) limb in st2_down for Turbidity FNU
Poor fit for rising (R²=0.16) or falling (R²=0.03) limb in st4_down for SSC (mg/L)
Poor fit for rising (R²=0.36) or falling (R²=0.52) limb in st4_up for SSC (mg/L)
Could not fit falling limb in st5_down for SSC (mg/L)
Could not fit falling limb in st5_down for Turbidity FNU
Poor fit for rising (R²=0.20) or falling (R²=0.00) limb in st5_up for SSC (mg/L)
Could not fit risi

# Spring Event Langlois HI 

In [15]:
event_directory = '../../data/constituents/spring_events'
events = {}
for filename in os.listdir(event_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(event_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        events[key] = df                                    # store the data frame in the dictionary

shear_stress = pd.read_csv('../../data/shear_stress/average_total_shear_stress_corrected.csv', parse_dates=['datetime'], index_col='datetime')
sonde_downstream = pd.read_csv('../../data/sonde_data/sonde_down_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')
sonde_upstream = pd.read_csv('../../data/sonde_data/sonde_up_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')

# only keep positive values in sonde data
sonde_downstream = sonde_downstream[(sonde_downstream["Turbidity FNU"] > 0) & (sonde_downstream["fDOM RFU"] > 0)]
sonde_upstream = sonde_upstream[(sonde_upstream["Turbidity FNU"] > 0) & (sonde_upstream["fDOM RFU"] > 0)]

# process and merge data
# average rows with duplicate timestamps on sonde data (second-resolution data)
sonde_downstream = sonde_downstream.groupby(level=0).mean(numeric_only=True)
sonde_upstream = sonde_upstream.groupby(level=0).mean(numeric_only=True)

# re-sample shear stress and sonde data to 1 min intervals
shear_stress = shear_stress.resample('1min').interpolate()
sonde_downstream_resampled = sonde_downstream.resample('1min').interpolate()
sonde_upstream_resampled = sonde_upstream.resample('1min').interpolate()

In [16]:
# join shear stress data and the matching sonde data with the storm data
merged_events = {}

for event_name, event_df in events.items():
    if "down" in event_name.lower():
        sonde_df = sonde_downstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    elif "up" in event_name.lower():
        sonde_df = sonde_upstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    else:
        continue

    merged_event_df = event_df.join(shear_stress, how="left")
    merged_event_df = merged_event_df.join(sonde_df, how="left")
    merged_events[event_name] = merged_event_df

# also join the sonde data with the shear stress data for the entire record 
sonde_downstream = sonde_downstream.join(shear_stress['shear_stress'], how="left")
sonde_upstream = sonde_upstream.join(shear_stress['shear_stress'], how="left")

In [17]:
merged_events['up_event1']

,SS (uL/L),SSC (mg/L),DOC (mg/L),shear_stress,Turbidity FNU,fDOM RFU
Date_Time,,,,,,
2023-04-17 16:30:00,103.31,16.000000,3.829,110.746094,8.384255,0.291897
2023-04-17 20:30:00,65.48,0.500000,3.473,113.229799,17.909117,0.910885
2023-04-18 00:30:00,73.68,6.315789,3.421,111.832163,12.502628,0.881094
2023-04-18 04:30:00,48.90,NaN,3.346,110.527790,12.258318,0.816276
2023-04-18 08:30:00,0.00,10.000000,3.271,109.388847,12.736497,0.762438


Calculate HI for all events

In [ ]:
all_results = []

for event_name, event_df in merged_events.items():
    for constituent in ["SSC (mg/L)", "Turbidity FNU"]:
        if constituent not in event_df.columns:
            continue

        result = compute_langlois_H(
            event_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=event_name)

        if result is not None:
            result["event"] = event_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('spring_events/langlois_hysteresis_spring.csv', index=False)

Poor fit for rising (R²=0.82) or falling (R²=0.46) limb in down_event1 for Turbidity FNU
Could not fit falling limb in down_event11 for SSC (mg/L)
Could not fit falling limb in down_event11 for Turbidity FNU
Could not fit rising limb in down_event12 for SSC (mg/L)
Could not fit rising limb in down_event12 for Turbidity FNU
Could not fit rising limb in down_event14 for SSC (mg/L)
Poor fit for rising (R²=1.00) or falling (R²=0.06) limb in down_event14 for Turbidity FNU
Poor fit for rising (R²=0.42) or falling (R²=1.00) limb in down_event2 for Turbidity FNU
Could not fit rising limb in down_event3 for SSC (mg/L)
Could not fit rising limb in down_event3 for Turbidity FNU
Could not fit rising limb in down_event4 for Turbidity FNU
No valid shear_stress or Turbidity FNU for down_event5; skipping
Poor fit for rising (R²=1.00) or falling (R²=0.13) limb in down_event7 for SSC (mg/L)


C:\Users\nicol\AppData\Local\Temp\ipykernel_37052\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Poor fit for rising (R²=0.51) or falling (R²=0.01) limb in down_event8 for SSC (mg/L)
Poor fit for rising (R²=0.78) or falling (R²=0.09) limb in down_event9 for SSC (mg/L)
Poor fit for rising (R²=0.31) or falling (R²=0.01) limb in down_event9 for Turbidity FNU
Poor fit for rising (R²=1.00) or falling (R²=0.17) limb in up_event14 for SSC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.07) limb in up_event2 for SSC (mg/L)
Poor fit for rising (R²=0.44) or falling (R²=0.96) limb in up_event2 for Turbidity FNU
Poor fit for rising (R²=1.00) or falling (R²=0.00) limb in up_event3 for SSC (mg/L)
Could not fit rising limb in up_event4 for SSC (mg/L)
Could not fit rising limb in up_event4 for Turbidity FNU
Could not fit falling limb in up_event5 for SSC (mg/L)
Poor fit for rising (R²=0.07) or falling (R²=1.00) limb in up_event5 for Turbidity FNU
Poor fit for rising (R²=0.03) or falling (R²=0.98) limb in up_event6 for Turbidity FNU
Poor fit for rising (R²=1.00) or falling (R²=0.32) limb in 

In [19]:
all_results = []

for event_name, event_df in merged_events.items():
    for constituent in ["SSC (mg/L)", "Turbidity FNU"]:
        if constituent not in event_df.columns:
            continue

        plot_langlois_hysteresis(
            event_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=event_name,
            out_dir='plots/langlois')

Poor fit for rising (R²=0.82) or falling (R²=0.46) limb in down_event1 for Turbidity FNU


C:\Users\nicol\AppData\Local\Temp\ipykernel_37052\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Could not fit falling limb in down_event11 for SSC (mg/L)
Could not fit falling limb in down_event11 for Turbidity FNU
Could not fit rising limb in down_event12 for SSC (mg/L)
Could not fit rising limb in down_event12 for Turbidity FNU
Could not fit rising limb in down_event14 for SSC (mg/L)
Poor fit for rising (R²=1.00) or falling (R²=0.06) limb in down_event14 for Turbidity FNU
Poor fit for rising (R²=0.42) or falling (R²=1.00) limb in down_event2 for Turbidity FNU
Could not fit rising limb in down_event3 for SSC (mg/L)
Could not fit rising limb in down_event3 for Turbidity FNU
Could not fit rising limb in down_event4 for Turbidity FNU
Poor fit for rising (R²=1.00) or falling (R²=0.13) limb in down_event7 for SSC (mg/L)
Poor fit for rising (R²=0.51) or falling (R²=0.01) limb in down_event8 for SSC (mg/L)
Poor fit for rising (R²=0.78) or falling (R²=0.09) limb in down_event9 for SSC (mg/L)
Poor fit for rising (R²=0.31) or falling (R²=0.01) limb in down_event9 for Turbidity FNU
Poor fi